# ARK-ASR-3B on our own recordings

[`Audio8/ARK-ASR-3B`](https://huggingface.co/Audio8/ARK-ASR-3B) tops the Open ASR
Leaderboard English short-form benchmark at **5.04% WER**. That benchmark is
AMI / Earnings22 / GigaSpeech / LibriSpeech / SPGISpeech / VoxPopuli — mostly
read or broadcast American and British English, mostly clean.

Our audio is none of those. It is Indian-accented spontaneous English, spoken at
a laptop microphone, recorded around −35 dBFS, full of `uh`/`um`, self-corrections
and proper nouns the model has never seen (Zoho, Mauritius, Chennai). A model can
win the leaderboard and still lose on this. So this notebook runs ARK-ASR-3B over
the recordings in `recordings/` and puts each **player next to its transcript**,
so you can listen and read at the same time and judge for yourself.

Where a recording has a sidecar `.jsonl` from the live demo, the streaming
recogniser's own output is shown alongside as a baseline — that is the transcript
the product produces today, and the number ARK has to beat to be worth switching to.

**What to look at**

- proper nouns and place names — where cloud-scale ASR usually falls over on our audio
- the `uh`/`um` handling — ARK transcribes them, the streaming model transcribes them differently
- RTFx — ARK is offline batch; the streaming model is realtime. They are not interchangeable.

## 1 · Setup

In [3]:
import gc, json, time, html
from pathlib import Path

import numpy as np
import librosa
import soundfile as sf
import torch
from IPython.display import Audio, HTML, display

MODEL_ID   = "Audio8/ARK-ASR-3B"
SR         = 16_000          # the model is 16 kHz only
MAX_WIN_S  = 28.0            # see the note on the 30 s encoder cap in §3
BATCH      = 4               # windows per generate() call
PROMPT     = "Please transcribe this audio."

REC = Path("recordings")
assert REC.is_dir(), f"run this notebook from demo-v3/, not {Path.cwd()}"


def pick_device() -> str:
    """Pick the GPU with the most free memory — this box shares its A6000s."""
    if not torch.cuda.is_available():
        return "cpu"
    free = [torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())]
    best = int(np.argmax(free))
    print(" ".join(f"cuda:{i} {f/2**30:.1f} GiB free" for i, f in enumerate(free)))
    if free[best] < 10 * 2**30:
        print(f"\n!! cuda:{best} has only {free[best]/2**30:.1f} GiB free; "
              "the model needs ~8.5 GiB in bf16 plus activations.")
    return f"cuda:{best}"


DEVICE = pick_device()
DTYPE  = torch.bfloat16 if DEVICE.startswith("cuda") else torch.float32
print("\nusing", DEVICE, DTYPE)

AssertionError: run this notebook from demo-v3/, not /workspace/speech_demo/demo-v3/notebooks

## 2 · Load the model

ARK-ASR is a Whisper-style encoder → MLP adapter → Qwen2 decoder, shipped as
`custom_code`, so `trust_remote_code=True` is required. ~8.1 GB of weights; the
first run downloads them to the HF cache.

**One gotcha worth the comment:** the decoder is causal, but the shipped processor
pads the text side *right* by default. Batch more than one window and every sample
but the longest gets its prompt trailed by padding, and generation comes back
**empty** — silently, no error. Setting `padding_side="left"` fixes it. Leave the
next line out and §5 returns blank transcripts for three windows out of five.

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor, AutoTokenizer

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Causal decoder + batched generate() ⇒ must left-pad. Right-padding returns "".
tokenizer.padding_side = "left"
processor.tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=DTYPE,
    attn_implementation="sdpa",
).to(DEVICE).eval()

print(f"loaded in {time.time()-t0:.1f}s · "
      f"{sum(p.numel() for p in model.parameters())/1e9:.2f}B params")


def _bad_words_ids(tok):
    """Block every special/angle-bracket token except EOS.

    Without this the decoder happily emits `<|im_start|>` and friends into the
    middle of a transcript. Straight from the model card.
    """
    eos  = tok.eos_token_id
    keep = {eos} if isinstance(eos, int) else set(eos or [])
    bad  = set(tok.all_special_ids) - keep
    bad.update(i for t, i in tok.get_added_vocab().items()
               if t.startswith("<") and t.endswith(">") and i not in keep)
    return [[i] for i in sorted(bad)]


BAD_WORDS = _bad_words_ids(tokenizer)

## 3 · Transcription helpers

### Why windowing is not optional

`config.json` sets `max_whisper_length: 1500` — 1500 encoder frames at a 10 ms hop
is **exactly 30 seconds**. Hand the processor anything longer and it truncates to
the first 30 s and transcribes only that, with no warning. Our agent sessions run
1–5 minutes, so anything past the first half-minute would simply vanish.

So `split_windows` chops audio into ≤28 s pieces, cutting at the **quietest point**
in the back half of each window rather than at a fixed offset — that lands the cut
in a pause instead of mid-word. It is a hard cap, not a silence detector: these
recordings have enough mic hiss that `librosa.effects.split` happily calls a whole
83 s session one continuous segment, which puts you right back at the 30 s cliff.

In [ ]:
def split_windows(y: np.ndarray, sr: int = SR, max_s: float = MAX_WIN_S):
    """Cut `y` into ≤ max_s windows, breaking at the quietest available point.

    Returns a list of (start_sample, end_sample). Short audio → one window.
    """
    n, max_n = len(y), int(max_s * sr)
    if n <= max_n:
        return [(0, n)]

    hop = int(0.02 * sr)                                    # 20 ms frames
    rms = librosa.feature.rms(y=y, frame_length=2 * hop, hop_length=hop)[0]

    bounds, s = [0], 0
    while n - s > max_n:
        lo, hi = s + int(0.55 * max_n), s + max_n           # search the back 45%
        seg = rms[lo // hop: hi // hop]
        s = (lo // hop + int(np.argmin(seg))) * hop if len(seg) else hi
        bounds.append(s)
    bounds.append(n)
    return list(zip(bounds[:-1], bounds[1:]))


@torch.inference_mode()
def _generate(arrays, max_new_tokens: int = 440):
    """Greedy-decode a list of ≤30 s float32 arrays. Returns a list of strings."""
    out = []
    for i in range(0, len(arrays), BATCH):
        chunk = arrays[i:i + BATCH]
        convs = [[{"role": "user", "content": [
                    {"type": "audio", "array": a},
                    {"type": "text",  "text": PROMPT}]}] for a in chunk]

        inputs = processor.apply_chat_template(
            convs,
            add_generation_prompt=True,
            return_tensors="pt",
            sampling_rate=SR,
            audio_padding="longest",
            text_kwargs={"padding": "longest"},
            audio_max_length=30 * SR,
        ).to(DEVICE)
        if "audios" in inputs:
            inputs["audios"] = inputs["audios"].to(dtype=DTYPE)

        ids = model.generate(
            **inputs,
            do_sample=False,                # deterministic; this is a measurement
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            bad_words_ids=BAD_WORDS,
        )
        out += [t.strip() for t in tokenizer.batch_decode(
            ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)]
    return out


def transcribe(source, normalise: bool = True):
    """Transcribe a wav path or a float array. Returns a dict of results.

    normalise: peak-normalise before the encoder. Our recordings sit around
    −35 dBFS; the mel front-end is not scale-invariant in principle, though in
    practice on this material it changes nothing — flip it off and compare.
    """
    if isinstance(source, (str, Path)):
        y, _ = librosa.load(str(source), sr=SR, mono=True)
    else:
        y = np.asarray(source, dtype=np.float32).reshape(-1)

    if normalise and len(y):
        y = y / max(np.abs(y).max(), 1e-6) * 0.9

    wins = split_windows(y)
    t0   = time.time()
    texts = _generate([y[a:b] for a, b in wins])
    wall  = time.time() - t0
    dur   = len(y) / SR

    return {
        "audio":    y,
        "duration": dur,
        "wall":     wall,
        "rtfx":     dur / wall if wall else float("inf"),
        "segments": [{"start": a / SR, "end": b / SR, "text": t}
                     for (a, b), t in zip(wins, texts)],
        "text":     " ".join(t for t in texts if t).strip(),
    }

### The streaming baseline

Every `agent-*` / `capture-*` / `kb-*` recording has a sidecar `.jsonl` written by
the live demo, one row per 80 ms chunk. `asr_text` there is the streaming
recogniser's hypothesis, *cumulative within a turn* and reset when the turn ends —
so the final value before each reset is that turn's finished transcript. That is
what the product ships today.

In [ ]:
def streaming_turns(jsonl_path) -> list[str]:
    """Recover the live recogniser's finished per-turn transcripts from a sidecar."""
    turns, prev = [], ""
    for line in Path(jsonl_path).read_text().splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        if row.get("type") != "chunk":
            continue
        text = (row.get("asr_text") or "").strip()
        if not text:                     # blank ⇒ the turn closed
            if prev:
                turns.append(prev)
                prev = ""
            continue
        if prev and not text.startswith(prev):   # hypothesis restarted
            turns.append(prev)
        prev = text
    if prev:
        turns.append(prev)

    merged: list[str] = []                       # collapse prefix duplicates
    for t in turns:
        if merged and (t.startswith(merged[-1]) or merged[-1].startswith(t)):
            merged[-1] = max(merged[-1], t, key=len)
        else:
            merged.append(t)
    return merged

### One recording, one cell

`listen(name)` is the whole point of the notebook: it prints the stats, renders a
**player**, and lays the ARK **transcript** out window by window underneath, with
the streaming baseline beside it when there is one. Play the clip, read across.

In [ ]:
def _resolve(name):
    """Accept a bare stem, a clips/ name, or a full path."""
    p = Path(name)
    for cand in (p, REC / name, REC / f"{name}.mic.wav", REC / "clips" / f"{name}.wav"):
        if cand.is_file():
            return cand
    raise FileNotFoundError(name)


def _esc(t):
    return html.escape(t) if t else '<span style="opacity:.35">— nothing —</span>'


def listen(name, normalise: bool = True, baseline: bool = True):
    """Player + ARK transcript (+ streaming baseline) for one recording."""
    path = _resolve(name)
    res  = transcribe(path, normalise=normalise)

    print(f"{path.name}  ·  {res['duration']:.1f}s audio  ·  "
          f"{len(res['segments'])} window(s)  ·  {res['wall']:.2f}s on {DEVICE}  ·  "
          f"RTFx {res['rtfx']:.0f}")
    display(Audio(res["audio"], rate=SR))

    rows = "".join(
        f"<tr><td style='padding:4px 12px 4px 0;white-space:nowrap;opacity:.55;"
        f"font-variant-numeric:tabular-nums'>{s['start']:6.1f}–{s['end']:6.1f}s</td>"
        f"<td style='padding:4px 0'>{_esc(s['text'])}</td></tr>"
        for s in res["segments"])
    display(HTML(
        "<div style='font:13px/1.55 ui-monospace,SFMono-Regular,Menlo,monospace'>"
        "<b>ARK-ASR-3B</b>"
        f"<table style='border-collapse:collapse;margin:6px 0 2px'>{rows}</table></div>"))

    side = path.with_name(path.name.replace(".mic.wav", "").replace(".wav", "") + ".jsonl")
    if baseline and side.is_file():
        turns = streaming_turns(side)
        if turns:
            items = "".join(f"<li style='margin:2px 0'>{html.escape(t)}</li>" for t in turns)
            display(HTML(
                "<div style='font:13px/1.55 ui-monospace,SFMono-Regular,Menlo,monospace;"
                "opacity:.72'><b>streaming recogniser (what we ship today)</b>"
                f"<ol style='margin:6px 0 2px;padding-left:22px'>{items}</ol></div>"))
    return res

## 4 · Short clips

`recordings/clips/` holds four hand-cut excerpts — the moments where the live
demo's turn-taking got it wrong, kept because they are hard. They are 9–29 s, so
each fits in a single window and there is no chunking in play: this is ARK on the
raw acoustic problem, nothing else.

**`01_malaysia-maurit__settle`** — Two place names, one of them uncommon.

In [ ]:
_ = listen("clips/01_malaysia-maurit__settle.wav")

**`02_yes-from-chen__confidence`** — A city name at the end of a trailing-off turn.

In [ ]:
_ = listen("clips/02_yes-from-chen__confidence.wav")

**`03_it-would-be-under__settle`** — Heavy self-correction; the speaker restarts three times.

In [ ]:
_ = listen("clips/03_it-would-be-under__settle.wav")

**`04_small__confidence`** — Quiet, and a proper noun the model has to guess at.

In [ ]:
_ = listen("clips/04_small__confidence.wav")

## 5 · Full agent sessions

Now the real thing: whole 1–5 minute sessions off the laptop mic. These exercise
the windowing, and each one has a streaming baseline underneath to read against.

**`agent-20260814-120139`** — 83 s, four turns of trip planning. This is the one that exposes the left-padding bug: right-pad the batch and windows 1–3 come back empty while window 4 looks fine, so the transcript reads as if the session started at 54 s.

In [ ]:
_ = listen("agent-20260814-120139")

**`capture-20260813-105609`** — 190 s. Opens with *“this is Vippin from Zoho Corporation of India”* — a personal name and a company name in the first six words, which is where most ASR breaks on us.

In [ ]:
_ = listen("capture-20260813-105609")

**`agent-20260819-080028`** — 218 s, twelve turns. The longest sustained back-and-forth we have.

In [ ]:
_ = listen("agent-20260819-080028")

**`agent-20260819-094920`** — 277 s. Mauritius / Chennai / calendar dates — the entity soup a travel agent actually hears. Note that most windows come back as a bare `uh`: this is the *mic* track, so the long gaps are the agent talking, not silence. `.ref.wav` is the other side.

In [ ]:
_ = listen("agent-20260819-094920")

## 6 · Sweep everything

The cells above are the ones worth listening to one at a time. This runs ARK over
every recording that has a streaming baseline to compare against, and prints one
table: duration, RTFx, and both transcripts truncated. Use it to spot the
disagreements, then come back and `listen()` to those.

Adjust `LIMIT` / `MIN_SEC` to taste — the full set is ~45 recordings and takes a
few minutes.

In [ ]:
LIMIT, MIN_SEC = 12, 20.0

candidates = []
for wav in sorted(REC.glob("*.mic.wav")):
    side = wav.with_name(wav.name.replace(".mic.wav", ".jsonl"))
    if not side.is_file():
        continue
    try:
        dur = sf.info(str(wav)).duration
    except Exception:
        continue
    if dur < MIN_SEC:
        continue
    turns = streaming_turns(side)
    if len(turns) >= 2:
        candidates.append((len(turns), dur, wav, turns))

candidates.sort(key=lambda c: -c[0])
print(f"{len(candidates)} recordings with a usable baseline; running {min(LIMIT, len(candidates))}\n")

results = []
for n_turns, dur, wav, turns in candidates[:LIMIT]:
    res = transcribe(wav)
    results.append({"name": wav.name.replace(".mic.wav", ""), "duration": dur,
                    "rtfx": res["rtfx"], "ark": res["text"],
                    "streaming": " ".join(turns), "n_turns": n_turns})
    print(f"  {res['rtfx']:6.0f}x  {dur:6.1f}s  {wav.name}")

def _cell(t, n=260):
    t = t or ""
    return html.escape(t[:n] + ("…" if len(t) > n else ""))

rows = "".join(
    f"<tr style='border-top:1px solid rgba(128,128,128,.25);vertical-align:top'>"
    f"<td style='padding:8px 12px 8px 0;white-space:nowrap'>{r['name']}<br>"
    f"<span style='opacity:.5'>{r['duration']:.0f}s · {r['rtfx']:.0f}x</span></td>"
    f"<td style='padding:8px 12px 8px 0'>{_cell(r['ark'])}</td>"
    f"<td style='padding:8px 0;opacity:.7'>{_cell(r['streaming'])}</td></tr>"
    for r in results)
display(HTML(
    "<table style='border-collapse:collapse;font:12px/1.5 ui-monospace,SFMono-Regular,Menlo,monospace'>"
    "<tr style='text-align:left'><th style='padding-right:12px'>recording</th>"
    "<th style='padding-right:12px'>ARK-ASR-3B</th><th>streaming (today)</th></tr>"
    f"{rows}</table>"))

## 7 · Optional — Whisper large-v3 as a third opinion

Two transcripts disagreeing tells you they disagree, not which one is right. When
ARK and the streaming model differ on a proper noun, a third independent model
casting a vote is usually enough to settle it. `openai/whisper-large-v3` is already
in the HF cache on this box.

This is a **sanity check, not a benchmark** — none of these recordings have a human
reference transcript, so there is no WER to compute here. For real numbers on
accented conversational English, `scripts/bench_asr.py` scores against EdAcc.

In [ ]:
def compare(name):
    """ARK vs Whisper large-v3 vs the live streaming model, on one recording."""
    from transformers import pipeline

    global _whisper
    try:
        _whisper
    except NameError:
        # Not on DEVICE: ARK is already holding ~8.5 GiB there, and large-v3 wants
        # another ~3. Landing both on one A6000 fails inside cuDNN with an
        # unhelpful "unable to find an engine" rather than a clean OOM.
        _whisper = pipeline("automatic-speech-recognition",
                            model="openai/whisper-large-v3",
                            torch_dtype=torch.float16, device=pick_device())

    path = _resolve(name)
    ark  = transcribe(path)
    wsp  = _whisper(str(path), chunk_length_s=30,
                    generate_kwargs={"language": "en"})["text"].strip()

    side  = path.with_name(path.name.replace(".mic.wav", "").replace(".wav", "") + ".jsonl")
    live  = " ".join(streaming_turns(side)) if side.is_file() else ""

    display(Audio(ark["audio"], rate=SR))
    rows = "".join(
        f"<tr style='border-top:1px solid rgba(128,128,128,.25);vertical-align:top'>"
        f"<td style='padding:8px 14px 8px 0;white-space:nowrap;font-weight:600'>{lbl}</td>"
        f"<td style='padding:8px 0'>{html.escape(txt) if txt else '—'}</td></tr>"
        for lbl, txt in [("ARK-ASR-3B", ark["text"]),
                         ("whisper-large-v3", wsp),
                         ("streaming (today)", live)])
    display(HTML(f"<div style='font:13px/1.55 ui-monospace,Menlo,monospace'>"
                 f"<table style='border-collapse:collapse'>{rows}</table></div>"))


compare("clips/01_malaysia-maurit__settle.wav")

In [ ]:
compare("clips/04_small__confidence.wav")

## 8 · What this run showed

From one pass over the recordings above, on an A6000 in bf16:

**Speed** — 25–150× realtime on full sessions, including windowing and mel
extraction, at batch 4. A
4½-minute session transcribes in about 2 seconds. This is offline batch decoding;
it says nothing about first-token latency, and ARK cannot replace the streaming
recogniser in the live loop. It is a candidate for post-call transcripts, not for
the turn-taking path.

**Proper nouns — the interesting part.** On our audio ARK is clearly ahead of the
streaming model on exactly the tokens we care about:

| said | streaming (today) | ARK-ASR-3B |
| --- | --- | --- |
| Gauhati | *"cohaki" / "Gauhaki"* | **gauhati** |
| Vidula Sri | *"Bitullah Street"* | **vidula sri** |
| Arul Vaidyan | *"Arul Vindan"* | **arul vaidyan** |
| Trivandrum | *"Tirupur"* | **trivandrum** |
| Coimbatore | *"Kwambatur"* | *"kwaimbatour"* ✗ |
| Mauritius | *"malicious" / "Malaysia"* | **mauritius** |

Neither model is clean, but ARK's errors are phonetically closer and it recovers
the harder Indian place and person names the streaming model mangles outright.

**Where it still loses.** It drops the sentence-final word on trailing-off turns
(`02_yes-from-chen`: *"yes from july"* where the speaker said Chennai), and on the
short hard clips in §4 it is no better than whisper-large-v3 — sometimes worse
(*"not malaysia or aches"* vs whisper's correct *"Not Malaysia, Mauritius"*).
Casing and punctuation are inconsistent: some windows come back fully lowercase,
others sentence-cased, which matters if anything downstream does entity extraction.

**What this is not.** None of these recordings have a human reference transcript,
so nothing here is a WER number — it is a listen-and-read comparison. To put a
figure on accented conversational English, `scripts/bench_asr.py` scores against
EdAcc, and adding ARK there is the obvious next step.

## 9 · Cleanup

Free the GPU when you are done — the A6000s on this box are shared.

In [ ]:
for _name in ("model", "_whisper"):
    if _name in dir():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()
print("released")